### 3.2.4. Nonsmooth Optimization

$$
\min_{\mathbf{x}} f(\mathbf{x}),
\qquad
\partial f(\mathbf{x}) = \{\mathbf{g} : f(\mathbf{y}) \ge f(\mathbf{x}) + \mathbf{g}^\top(\mathbf{y}-\mathbf{x})\ \forall \mathbf{y}\},
$$
with optimality condition $\mathbf 0 \in \partial f(\mathbf{x}^\star)$.

**Explanation:**

Many useful objectives — the absolute value, the $\ell_1$ norm, a pointwise maximum — are convex but not differentiable at some points. The gradient is replaced by the *subdifferential* $\partial f$, the set of all subgradient vectors whose linear underestimator touches the graph at $\mathbf{x}$. A point is optimal exactly when the zero vector lies in the subdifferential, generalizing $\nabla f(\mathbf{x}^\star)=\mathbf 0$. A nonsmooth problem can often be turned into a smooth [linear](../04_Convex_Problem_Classes/01_linear_programming.ipynb) or [second-order cone](../04_Convex_Problem_Classes/04_second_order_cone_programming.ipynb) program by introducing epigraph variables.

**Numerical Example:**

Minimize the sum of absolute deviations
$$
f(x) = |x - 1| + |x + 2|.
$$

For $x\in[-2,1]$ the two terms give $(1-x)+(x+2)=3$, a constant, so every point of the interval is a minimizer with value $3$. At an interior point such as $x=0$ the subdifferentials are
$$
\partial|x-1|\big|_{0} = \{-1\},\quad \partial|x+2|\big|_{0} = \{+1\},
\quad\Rightarrow\quad 0 \in \partial f(0) = \{0\}.
$$

The zero subgradient certifies optimality even though $f$ is not differentiable at the kinks $x=1$ and $x=-2$.

In [1]:
import sympy as sp

x = sp.symbols("x", real=True)
objective = sp.Abs(x - 1) + sp.Abs(x + 2)

subgradient_at_zero = sp.diff(objective, x).subs(x, 0)
sample_grid = [sp.Integer(v) for v in range(-4, 4)]
values = [(v, objective.subs(x, v)) for v in sample_grid]

print("objective values on grid =", values)
print("derivative at x = 0 (interior of minimizers) =", subgradient_at_zero)
print("minimum value =", min(value for _, value in values))

objective values on grid = [(-4, 7), (-3, 5), (-2, 3), (-1, 3), (0, 3), (1, 3), (2, 5), (3, 7)]
derivative at x = 0 (interior of minimizers) = 0
minimum value = 3


**Equivalent casadi implementation:**

In [2]:
import casadi as ca

x = ca.SX.sym("x")
t_1 = ca.SX.sym("t_1")
t_2 = ca.SX.sym("t_2")
decision_variable = ca.vertcat(x, t_1, t_2)

objective = t_1 + t_2
epigraph = ca.vertcat(t_1 - (x - 1), t_1 + (x - 1), t_2 - (x + 2), t_2 + (x + 2))

solver = ca.nlpsol("solver", "ipopt", {"x": decision_variable, "f": objective, "g": epigraph},
                   {"ipopt.print_level": 0, "print_time": 0, "ipopt.sb": "yes"})
solution = solver(x0=[0, 5, 5], lbg=0, ubg=ca.inf)

print("minimizer x =", float(solution["x"][0]))
print("minimum value =", float(solution["f"]))

minimizer x = -0.40208353994282964
minimum value = 2.999999985009544


**References:**

[📘 Boyd, S., & Vandenberghe, L. (2004). *Convex Optimization*. Cambridge University Press.](https://web.stanford.edu/~boyd/cvxbook/)  
[📘 Bertsekas, D. P. (1999). *Nonlinear Programming* (2nd ed.). Athena Scientific.](https://www.athenasc.com/nonlinbook.html)

---

[⬅️ Previous: Smooth Optimization](./03_smooth_optimization.ipynb) | [Next: Nonlinear Programming ➡️](./05_nonlinear_programming_overview.ipynb)